In [ ]:
%pip install opencv-python imageio mediapipe kagglehub

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# import torchvision

import cv2
# import tensorflow as tf

import numpy as np
from typing import List
from matplotlib import pyplot as plt
import imageio
from IPython.display import Image, display


In [2]:
from processing.video_loader import VideoLoader
from processing.preprocessor import Preprocessor

# building data loading functions

download dataset from kaggle
https://www.kaggle.com/datasets/mohamedbentalb/lipreading-dataset and save them in data\lips-dataset

In [ ]:
%pip install -q kagglehub

import shutil
from pathlib import Path
import kagglehub

DATASET_HANDLE = 'mohamedbentalb/lipreading-dataset'
target_dir = Path('data/lips-dataset')

base = Path(kagglehub.dataset_download(DATASET_HANDLE))
for candidate in (base, base / 'data'):
    if (candidate / 's1').is_dir() or (candidate / 'alignments').is_dir():
        base = candidate
        break

print(f'Downloaded to: {base}')

target_dir.mkdir(parents=True, exist_ok=True)
for name in ('s1', 'alignments'):
    src = base / name
    dst = target_dir / name
    if not src.exists():
        print(f'SKIP: {name} not found in {base}')
    elif dst.exists():
        print(f'SKIP: {name} already at {dst}')
    else:
        shutil.move(str(src), str(dst))
        print(f'Moved {name} -> {dst}')


In [4]:
#test_path = r'.\data\lips-dataset\s1\bbaf2n.mpg' ## Windows File Directory
# test_path = r'./data/lips-dataset/s1/bbaf2n.mpg' ## Macbook File Directory
test_path = os.path.join("data", "lips-dataset", "s1", "bbaf2n.mpg") # testing for all directory
video_loader = VideoLoader()
video_data = video_loader.load(test_path)
# video_data

In [5]:
preprocessor = Preprocessor()
video_data = preprocessor.convert_to_greyscale(video_data)

if fail to crop lips, head over to google https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task



In [ ]:
# cv2.imshow("greyscale", video_data["frames"][0])
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [6]:
vocab = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]

In [7]:
char_to_num = {char: idx for idx, char in enumerate(vocab)}
num_to_char = {idx: char for char, idx in char_to_num.items()}

print(f"The vocabulary is: {vocab} (size = {len(vocab)})")

The vocabulary is: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', "'", '?', '!', '1', '2', '3', '4', '5', '6', '7', '8', '9', ' '] (size = 39)


In [8]:
def encode_text(text):
    return torch.tensor([char_to_num[c] for c in text], dtype=torch.long)

def decode_indices(indices):
    return "".join(num_to_char[int(i)] for i in indices)

x = encode_text("hello")
print(x)
print(decode_indices(x))

tensor([ 7,  4, 11, 11, 14])
hello


In [9]:
def load_alignments(path:str) -> List[str]: 
    with open(path, 'r') as f: 
        lines = f.readlines() 
    tokens = []
    for line in lines:
        line = line.split()
        if line[2] != 'sil': 
            tokens = [*tokens,' ',line[2]]
    return [encode_text(x) for x in tokens]

def load_data(path: str) -> dict: 
    # path = bytes.decode(path.numpy())
    #file_name = path.split('/')[-1].split('.')[0]
    # File name splitting for windows
    file_name = os.path.basename(path).split('.')[0]
    video_path = os.path.join('data','lips-dataset','s1',f'{file_name}.mpg')
    alignment_path = os.path.join('data', 'lips-dataset', 'alignments','s1',f'{file_name}.align')
    video_data = video_loader.load(path) 
    alignments = load_alignments(alignment_path)
    video_data['alignments'] = alignments
    return video_data


video_data = load_data(test_path)
# print(video_data)
print(video_data['alignments'])
print([decode_indices(x) for x in video_data['alignments']])

[tensor([38]), tensor([ 1,  8, 13]), tensor([38]), tensor([ 1, 11, 20,  4]), tensor([38]), tensor([ 0, 19]), tensor([38]), tensor([5]), tensor([38]), tensor([19, 22, 14]), tensor([38]), tensor([13, 14, 22])]
[' ', 'bin', ' ', 'blue', ' ', 'at', ' ', 'f', ' ', 'two', ' ', 'now']


# Creating data pipeline

In [ ]:
samples  = []
word_dict = {}
for i in os.listdir('data/lips-dataset/s1'):
    if i.endswith('.mpg'):
        print(i)
        video_data = load_data(os.path.join('data/lips-dataset/s1', i))
        samples.append((video_data['frames'], video_data['alignments']))
        
        print(video_data['alignments'])
        sentence = [decode_indices(x) for x in video_data['alignments']]
        print(sentence)
        for word in sentence: 
            if word not in word_dict: 
                word_dict[word] = 1
            else: 
                word_dict[word] += 1
    break

print(sorted(word_dict.items(), key=lambda x: x[1], reverse=True))

In [ ]:
#cv2.imshow("test", video_data['frames'][40])
#cv2.waitKey(0)
#cv2.destroyAllWindows()

## Alignment Algorithm
### LBY: The cell below runs the align_videos.py as if it was in a terminal, input arguments include data directory and output directory
### It is important to note to save the kaggle dataset in its subfolder called data/lips-dataset, of which should have the following subfolders:
### (a) alignments, (b) s1, (c) processedData [This is an addition]

In [ ]:
!python align_videos.py --video-dir data/lips-dataset/s1 --align-dir data/lips-dataset/alignments/s1 --output-dir data/lips-dataset/processedData --grayscale --crop-mouth

In [ ]:
import numpy as np
data = np.load(r'data\lips-dataset\processedData\bbaf2n.npz')
cv2.imshow('Frames', data['frames'][0])
cv2.waitKey(0)
cv2.destroyAllWindows()

## Train/Test Split & Model Evaluation
### LBY: Split `aligned_words.csv` (word-level, stratified 80/20), train a LipNet-style classifier, and report test accuracy.


In [10]:
from dataset import get_word_classification_dataloaders
from models.LBY.model import LipReadingClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_loader, test_loader, word_to_index, index_to_word = get_word_classification_dataloaders(
    test_size=0.2, seed=42, batch_size=32, stratify=True
)
num_classes = len(word_to_index)

print(f'Device: {device}')
print(f'Train samples: {len(train_loader.dataset)}')
print(f'Test samples: {len(test_loader.dataset)}')
print(f'Classes: {num_classes}')


Device: cuda
Train samples: 4796
Test samples: 1204
Classes: 51


In [11]:
train_loader.dataset.dataframe

,video_id,video_file,alignment_file,npz_file,word_index,word,start_units,end_units,start_seconds,end_seconds,start_frame,end_frame,frame_count,fps,npz_path
0,lrwz4p,lrwz4p.mpg,lrwz4p.align,lrwz4p.npz,3,z,26000,31250,1.04,1.25,26,31,5,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
1,lwbszn,lwbszn.mpg,lwbszn.align,lwbszn.npz,1,white,22250,29000,0.89,1.16,22,29,7,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
2,lbayzp,lbayzp.mpg,lbayzp.align,lbayzp.npz,0,lay,11250,19750,0.45,0.79,11,20,9,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
3,srbb7a,srbb7a.mpg,srbb7a.align,srbb7a.npz,3,b,33500,38000,1.34,1.52,34,38,4,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
4,sban4p,sban4p.mpg,sban4p.align,sban4p.npz,1,blue,21250,29250,0.85,1.17,21,29,8,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4791,bbbm1s,bbbm1s.mpg,bbbm1s.align,bbbm1s.npz,0,bin,14500,19500,0.58,0.78,15,20,5,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
4792,bwim6p,bwim6p.mpg,bwim6p.align,bwim6p.npz,2,in,25750,28500,1.03,1.14,26,29,3,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
4793,bgbo1a,bgbo1a.mpg,bgbo1a.align,bgbo1a.npz,2,by,34000,39500,1.36,1.58,34,40,6,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
4794,lgiz2n,lgiz2n.mpg,lgiz2n.align,lgiz2n.npz,0,lay,12750,19250,0.51,0.77,13,19,6,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...


In [12]:
test_loader.dataset.dataframe

,video_id,video_file,alignment_file,npz_file,word_index,word,start_units,end_units,start_seconds,end_seconds,start_frame,end_frame,frame_count,fps,npz_path
0,bwaa3a,bwaa3a.mpg,bwaa3a.align,bwaa3a.npz,5,again,45000,54750,1.80,2.19,45,55,10,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
1,lray6p,lray6p.mpg,lray6p.align,lray6p.npz,1,red,28500,33250,1.14,1.33,29,33,4,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
2,swbv4p,swbv4p.mpg,swbv4p.align,swbv4p.npz,1,white,17250,22250,0.69,0.89,17,22,5,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
3,pgij8n,pgij8n.mpg,pgij8n.align,pgij8n.npz,1,green,20500,26750,0.82,1.07,21,27,6,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
4,lbad9a,lbad9a.mpg,lbad9a.align,lbad9a.npz,2,at,26000,28000,1.04,1.12,26,28,2,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1199,bgbh5s,bgbh5s.mpg,bgbh5s.align,bgbh5s.npz,2,by,25750,29000,1.03,1.16,26,29,3,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
1200,lgas2n,lgas2n.mpg,lgas2n.align,lgas2n.npz,1,green,21000,24500,0.84,0.98,21,25,4,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
1201,lwbs3a,lwbs3a.mpg,lwbs3a.align,lwbs3a.npz,1,white,21250,31500,0.85,1.26,21,32,11,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...
1202,pgby7a,pgby7a.mpg,pgby7a.align,pgby7a.npz,4,seven,39750,48000,1.59,1.92,40,48,8,25.0,C:\Users\Valente\OneDrive\Documents\NUS\Patter...


In [13]:
from pathlib import Path

import torch
import torch.nn as nn

from models.LBY.train import set_seed, train_one_epoch, evaluate

set_seed(42)

model = LipReadingClassifier(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [14]:
out_dir = Path('checkpoints')
out_dir.mkdir(parents=True, exist_ok=True)

epochs = 2  
best_accuracy = 0.0
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(
        f'Epoch {epoch:2d}/{epochs} | '
        f'train_loss {train_loss:.4f} train_acc {train_acc:.4f} | '
        f'test_loss {test_loss:.4f} test_acc {test_acc:.4f}'
    )

    if test_acc > best_accuracy:
        best_accuracy = test_acc
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'word_to_index': word_to_index,
            'test_accuracy': test_acc,
        }
        # torch.save(checkpoint, out_dir / 'best1.pt')

print(f'Best test accuracy: {best_accuracy:.4f}')

Epoch  1/2 | train_loss 3.2853 train_acc 0.1082 | test_loss 2.9816 test_acc 0.1337
Epoch  2/2 | train_loss 3.0275 train_acc 0.1447 | test_loss 2.8896 test_acc 0.1462
Best test accuracy: 0.1462


In [15]:
# Optionally load the checkpoint saved above and re-evaluate.
checkpoint = torch.load('checkpoints/best.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
saved_epoch = checkpoint['epoch']
saved_acc = checkpoint['test_accuracy']
print(f'Loaded checkpoint (epoch {saved_epoch}, saved test acc {saved_acc:.4f})')
print(f'Re-evaluated test accuracy: {evaluate(model, test_loader, criterion, device)[1]:.4f}')


Loaded checkpoint (epoch 91, saved test acc 0.8522)
Re-evaluated test accuracy: 0.8530


# Testing with sample video

## Valente: predicting on a sample video. via the best model

In [16]:
test_path = os.path.join("data", "lips-dataset", "s1", "swwv7s.mpg") # testing for all directory


In [17]:
# 1. Load .mpg
video_loader = VideoLoader()
preprocessor = Preprocessor()
video_frames = video_loader.load(test_path)
preprocessed_frames = preprocessor.convert_to_greyscale(video_frames)

print(type(preprocessed_frames['frames']))

<class 'numpy.ndarray'>


In [18]:
# 3. Convert to tensor
batch = torch.from_numpy(preprocessed_frames['frames'].astype(np.float32)).unsqueeze(0).unsqueeze(2)  # → (1, T, 1, 50, 100)
# 4. Inference
model.eval()
with torch.no_grad():
    logits = model(batch.to(device))
    pred_idx = logits.argmax(dim=1).item()
    pred_word = index_to_word[pred_idx]

print(f"Predicted word: {pred_word}")

Predicted word: seven


In [19]:
test_path = np.load(os.path.join("data", "lips-dataset", "processedData", "swwv9a.npz"))['frames']

In [20]:
# 3. Convert to tensor
batch = torch.from_numpy(test_path.astype(np.float32)).unsqueeze(0).unsqueeze(2)  # → (1, T, 1, 50, 100)
# 4. Inference
model.eval()
with torch.no_grad():
    logits = model(batch.to(device))
    pred_idx = logits.argmax(dim=1).item()
    pred_word = index_to_word[pred_idx]

print(f"Predicted word: {pred_word}")

Predicted word: seven
